In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d
from natsort import natsorted
import pymeshlab

from plant_shape_analysis.dataloaders.trackplant3D import LeafSequencesDataset
from plant_shape_analysis.data_preprocessing.leaf_meshing import mesh_leaf

from plant_shape_analysis.reconstruction import classical_methods

%load_ext autoreload
%autoreload 2

## Mesh example single leaf

In [ ]:
dataset_path = Path("../data/TrackPlant3D")
leaf_dataset = LeafSequencesDataset(
    dataset_path, 
    min_timepoints=1, # we get all leaves
)

In [ ]:
maize_leaves = leaf_dataset.get_timeseries_by_crop('maize')
len(maize_leaves)

In [ ]:
seq_id = 0
timepoint = 0
leaf = maize_leaves[seq_id]['timepoints'][timepoint]
leaf

In [ ]:
pcd, mesh = mesh_leaf(leaf['points'], method="ball_pivoting", backend="pymeshlab")
mesh.get_surface_area()

In [ ]:
pcd, mesh = mesh_leaf(leaf['points'], method="ball_pivoting", backend="open3d")
mesh.get_surface_area()

In [ ]:
pcd, mesh = mesh_leaf(leaf['points'], method="poisson", backend="pymeshlab")
mesh.get_surface_area()

In [ ]:
pcd, mesh = mesh_leaf(leaf['points'], method="poisson", backend="open3d")
mesh.get_surface_area()

In [ ]:
o3d.visualization.draw_geometries(
[mesh],
point_show_normal=True,
window_name="Leaf Point Cloud and normals",
)

## Extract all dense leaves

In [ ]:
dataset_path = Path("../data/TrackPlant3D")
leaf_dataset = LeafSequencesDataset(
    dataset_path, 
    min_timepoints=1, # we get all leaves
)

In [ ]:
# for seq in leaf_dataset:
#     for scan in seq['timepoints']:
#         if scan['dense_points'] is not None:
#             # save leaf
#             crop = ''.join([c for c in seq['sequence_name'].split('_')[0] if c.isalpha()])
#             file_name = scan['file_path'].name.replace(".txt", f"_leaf{scan['leaf_id']}_dense.ply")
#             pcd = o3d.geometry.PointCloud()
#             pcd.points = o3d.utility.Vector3dVector(scan['dense_points'])
#             save_dir = Path(f"../data/TrackPlant3D/dense_leaves/{crop}")
#             save_dir.mkdir(parents=True, exist_ok=True)
#             o3d.io.write_point_cloud(save_dir / file_name, pcd)

In [ ]:
# import re
# import pandas as pd

# # Get file names
# dataset_path = Path("../data/TrackPlant3D/dense_leaves")
# files = []
# for crop in ['maize', 'tomato']:
#     crop = [f.stem for f in (dataset_path / crop).iterdir()]
#     files.extend(crop)
# files = natsorted(files)

# # Helper func
# def extract_numeric_suffix(x):
#     return int(re.search(r"\d+$", x).group(0))

# df = pd.DataFrame(files, columns=['file_name'])
# df['crop'] = df['file_name'].apply(lambda x: x.split('_')[1])
# df['treatment'] = df['file_name'].apply(lambda x: x.split('_')[2])
# df['plant'] = df['file_name'].apply(lambda x: extract_numeric_suffix(x.split('_')[3]))
# df['day'] = df['file_name'].apply(lambda x: extract_numeric_suffix(x.split('_')[4]))
# df['leaf'] = df['file_name'].apply(lambda x: extract_numeric_suffix(x.split('_')[5]))
# df['leaf_sequence_id'] = df.groupby(['crop', 'treatment','plant', 'leaf']).ngroup()
# df.insert(0, 'leaf_sequence_id', df.pop('leaf_sequence_id'))
# df['dense_leaf_id'] = df.index
# df.insert(0, 'dense_leaf_id', df.pop('dense_leaf_id'))
# df.to_csv("../data/TrackPlant3D/dense_leaves/dense_leaves_metadata.csv", index=False)

## Quality assurance and outlier removal

In [ ]:
idx = 60
leaf_paths = natsorted(Path("../data/TrackPlant3D/dense_leaves/maize").iterdir())
leaf_paths[idx]

In [ ]:
pcd = o3d.t.io.read_point_cloud(str(leaf_paths[idx]))
print(f"Loaded point cloud with {len(pcd.to_legacy().points)} points")
pcd.estimate_normals()
pcd.orient_normals_consistent_tangent_plane(k=30)  # This is very important!!
pcd.orient_normals_to_align_with_direction() # This seems even more important 
# o3d.visualization.draw_geometries([pcd.to_legacy()], point_show_normal=True, window_name="Original point cloud")

Outlier removal

In [ ]:
# statistical outlier removal
clean_pcd, clean_idx = pcd.to_legacy().remove_statistical_outlier(nb_neighbors=4, std_ratio=10)
removed_indices = np.setdiff1d(np.arange(len(pcd.to_legacy().points)), np.asarray(clean_idx))
print(f"Removed {len(removed_indices)} outlier points.")
#o3d.visualization.draw_geometries([clean_pcd], window_name="Cleaned point cloud")

In [ ]:
def display_inlier_outlier(cloud, ind, visualize_separately=False):
    inlier_cloud = cloud.select_by_index(ind)
    outlier_cloud = cloud.select_by_index(ind, invert=True)

    print("Showing outliers (red) and inliers (gray): ")
    outlier_cloud.paint_uniform_color([1, 0, 0])
    inlier_cloud.paint_uniform_color([0.8, 0.8, 0.8])
    o3d.visualization.draw_geometries([inlier_cloud, outlier_cloud], window_name="Inliers (grey) and outliers (red)")
    if visualize_separately:
        o3d.visualization.draw_geometries([inlier_cloud], window_name="Inlier cloud")
        o3d.visualization.draw_geometries([outlier_cloud], window_name="Outlier cloud")

display_inlier_outlier(pcd.to_legacy(), clean_idx, visualize_separately=False)

In [ ]:
#o3d.io.write_point_cloud("removed_points.ply", pcd.to_legacy().select_by_index(clean_idx, invert=True))

## Quick test classical meshing methods

In [ ]:
# Load and visualize a sample leaf point cloud
#sample_leaf_path = Path("../data/TrackPlant3D/dense_leaves/maize/19_maize_control_plant3_D00_leaf1_dense.ply")
sample_leaf_path = Path("../data/TrackPlant3D/examples/single_leafs/maize/1_maize_control_plant1_D00_leaf1_dense.ply")
#sample_leaf_path = Path("../data/TrackPlant3D/examples/single_leafs/maize/1_maize_control_plant1_D00_leaf1_dense_downsampled-voxel02.ply")
leaf_pcd = o3d.io.read_point_cloud(str(sample_leaf_path))
leaf_np = np.asarray(leaf_pcd.points)
print(f"Loaded leaf point cloud with {leaf_np.shape[0]} points.")

# Get normals too 
leaf_pcd.estimate_normals()
leaf_pcd.orient_normals_to_align_with_direction()
leaf_pcd.orient_normals_consistent_tangent_plane(k=50)
point_cloud = np.hstack([leaf_np, np.asarray(leaf_pcd.normals)])

o3d.visualization.draw_geometries([leaf_pcd])

### Apha shapes

In [ ]:
mesh_alpha = classical_methods.alpha_shape_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_alpha]) # -> Not bad but has holes
mesh_alpha.get_surface_area()

### Bezier

In [ ]:
mesh_bezier = classical_methods.bezier_surface_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_bezier]) # -> Has some issues, needs tuning
mesh_bezier.get_surface_area()

### B-spline

In [ ]:
mesh_bspline = classical_methods.bspline_surface_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_bspline]) # -> Looks better than bezier, but has some singularities
mesh_bspline.get_surface_area()

### Delaunay

In [ ]:
mesh_delaunay = classical_methods.delaunay_mesh_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_delaunay])
mesh_delaunay.get_surface_area()

### Poisson

In [ ]:
mesh_poisson = classical_methods.poisson_reconstruction(leaf_np, depth=9)
o3d.visualization.draw_geometries([mesh_poisson])
mesh_poisson.get_surface_area()

### Ball pivoting

In [ ]:
mesh_bpa = classical_methods.ball_pivoting_reconstruction(leaf_np, radius_multiplier=5.0)
o3d.visualization.draw_geometries([mesh_bpa])
mesh_bpa.get_surface_area()

Alternatively the area could be computed as below but it gives the same results

In [ ]:
vertices_bezier = np.asarray(mesh_bezier.vertices)
triangles_bezier = np.asarray(mesh_bezier.triangles)

area_bezier_manual = classical_methods.compute_mesh_area(vertices_bezier, triangles_bezier)
float(area_bezier_manual)

## Trimmed Poisson

In [ ]:
# Load and visualize a sample leaf point cloud
# sample_leaf_path = Path("../data/TrackPlant3D/dense_leaves/maize/11_maize_control_plant2_D01_leaf1_dense.ply")
# leaf_pcd = o3d.io.read_point_cloud(str(sample_leaf_path))

crop = 'maize' # 'tomato'
idx = 53
leaf_paths = natsorted(Path(f"../data/TrackPlant3D/dense_leaves/{crop}").iterdir())
leaf_pcd = o3d.io.read_point_cloud(str(leaf_paths[idx]))
leaf_np = np.asarray(leaf_pcd.points)

leaf_pcd.estimate_normals()
leaf_pcd.orient_normals_to_align_with_direction()
leaf_pcd.orient_normals_consistent_tangent_plane(k=50)
point_cloud = np.hstack([leaf_np, np.asarray(leaf_pcd.normals)])
print(f"Loaded leaf point cloud with {leaf_np.shape[0]} points.")
o3d.visualization.draw_geometries([leaf_pcd])

### Reconstruct Poisson surface

In [ ]:
#pcd, mesh = mesh_leaf(leaf_np, method="poisson", backend="pymeshlab")

With Meshlab

In [ ]:
# ms = pymeshlab.MeshSet()
# mesh = pymeshlab.Mesh(vertex_matrix=leaf_np)
# ms.add_mesh(mesh, "leaf")
# ms.compute_normal_for_point_clouds(k=20, smoothiter=10)
# ms.generate_surface_reconstruction_screened_poisson(depth=7)
# mesh = ms.current_mesh()

# vertices = np.asarray(mesh.vertex_matrix())
# triangles = np.asarray(mesh.face_matrix())

# o3d_mesh = o3d.geometry.TriangleMesh()
# o3d_mesh.vertices = o3d.utility.Vector3dVector(vertices)
# o3d_mesh.triangles = o3d.utility.Vector3iVector(triangles)
# o3d_mesh.compute_vertex_normals() # this makes faces look good
# o3d.visualization.draw_geometries([o3d_mesh])

Try poisson with open3d

In [ ]:
pcd = o3d.t.io.read_point_cloud(str(leaf_paths[idx]))
pcd.estimate_normals()
pcd.orient_normals_consistent_tangent_plane(k=500)  # This is very important!!
pcd.orient_normals_to_align_with_direction() # Putting this after consistent_tangent_plane seems to work better for normals to point outward from the inside face of the leaf
o3d.visualization.draw_geometries([pcd.to_legacy()], point_show_normal=True)

In [ ]:
# pcd = o3d.t.geometry.PointCloud()
# pcd.points = o3d.utility.Vector3dVector(leaf_np)

# # Estimate normals
# pcd.estimate_normals()
# pcd.orient_normals_to_align_with_direction()
# pcd.orient_normals_consistent_tangent_plane(k=30)  # This is very important!!

# Create mesh using Poisson Surface Reconstruction
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd.to_legacy(), depth=7, scale=1.1)
mesh.compute_vertex_normals()
mesh.orient_triangles()
#mesh.vertex_colors = o3d.utility.Vector3dVector(np.empty((0, 3)))

# Plot based on density
densities = np.asarray(densities)
density_colors = plt.cm.plasma(densities / np.max(densities))[:, :3]  # Use colormap
mesh.vertex_colors = o3d.utility.Vector3dVector(density_colors)
o3d.visualization.draw_geometries([mesh])

### Trimming based on density

In [ ]:
def apply_threshold_and_visualize(mesh, densities, threshold):
    """Apply threshold and show before/after comparison"""
    
    # Get original data
    vertices = np.asarray(mesh.vertices)
    triangles = np.asarray(mesh.triangles)
    
    # Create mask for vertices to keep
    vertices_to_keep = densities >= threshold
    
    # Create mapping from old indices to new indices
    vertex_mapping = np.full(len(vertices), -1)
    vertex_mapping[vertices_to_keep] = np.arange(np.sum(vertices_to_keep))
    
    # Filter vertices
    new_vertices = vertices[vertices_to_keep]
    
    # Filter triangles - keep only triangles where all vertices are kept
    triangle_mask = np.all(vertices_to_keep[triangles], axis=1)
    new_triangles = triangles[triangle_mask]
    
    # Remap triangle indices
    new_triangles = vertex_mapping[new_triangles]
    
    # Create new mesh
    trimmed_mesh = o3d.geometry.TriangleMesh()
    trimmed_mesh.vertices = o3d.utility.Vector3dVector(new_vertices)
    trimmed_mesh.triangles = o3d.utility.Vector3iVector(new_triangles)
    trimmed_mesh.compute_vertex_normals()

    
    # Copy colors if they exist
    if mesh.has_vertex_colors():
        vertex_colors = np.asarray(mesh.vertex_colors)
        new_vertex_colors = vertex_colors[vertices_to_keep]
        trimmed_mesh.vertex_colors = o3d.utility.Vector3dVector(new_vertex_colors)
    
    # Show statistics
    original_vertices = len(vertices)
    remaining_vertices = len(new_vertices)
    print(f"Original vertices: {original_vertices}")
    print(f"Remaining vertices: {remaining_vertices}")
    print(f"Removed: {original_vertices - remaining_vertices} ({100*(1-remaining_vertices/original_vertices):.1f}%)")
    
    # Visualize both
    print("Original mesh (press Q to close and see trimmed)")
    o3d.visualization.draw_geometries([mesh], window_name="Original Mesh")
    
    print("Trimmed mesh")
    o3d.visualization.draw_geometries([trimmed_mesh], window_name="Trimmed Mesh")
    
    return trimmed_mesh

In [ ]:
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(leaf_np)

# Estimate normals
pcd.estimate_normals()
pcd.orient_normals_to_align_with_direction()
pcd.orient_normals_consistent_tangent_plane(k=30)  # This is very important!!

# Create mesh using Poisson Surface Reconstruction
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=7, scale=1.1)
mesh.compute_vertex_normals()
mesh.vertex_colors = o3d.utility.Vector3dVector(np.empty((0, 3)))
o3d.visualization.draw_geometries([mesh])

In [ ]:
selected_threshold = np.percentile(densities, 25)  # Adjust based on your plots
trimmed_mesh = apply_threshold_and_visualize(mesh, densities, selected_threshold)
o3d.visualization.draw_geometries([pcd, trimmed_mesh], window_name="Trimmed Mesh")
# o3d.io.write_triangle_mesh("trimmed_leaf_mesh.ply", trimmed_mesh)
# pcd, mesh = mesh_leaf(leaf_np, method="poisson", backend="pymeshlab")

### Trimming based on density + distance

Option 1: based on alpha shape boundary

In [ ]:
def create_alpha_shape_boundary(pcd, alpha=0.1):
    """Create alpha shape to define better boundaries"""
    
    # Create alpha shape (2D or 3D)
    alpha_shape = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd, alpha)
    alpha_shape.compute_vertex_normals()
    
    return alpha_shape

def combine_poisson_with_alpha_boundary(pcd, poisson_depth=7, alpha=0.1, density_threshold_percentile=10):
    """Combine Poisson reconstruction with alpha shape boundary"""
    
    # Step 1: Poisson reconstruction
    poisson_mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd, depth=poisson_depth
    )
    
    # Step 2: Create alpha shape for boundary reference
    alpha_shape = create_alpha_shape_boundary(pcd, alpha)
    
    # Step 3: Filter Poisson mesh by density
    density_threshold = np.percentile(densities, density_threshold_percentile)
    density_mask = densities >= density_threshold

    # Step 4: Additional filtering - keep vertices close to alpha shape
    poisson_vertices = np.asarray(poisson_mesh.vertices)
    alpha_vertices = np.asarray(alpha_shape.vertices)
    
    # Find distances from Poisson vertices to alpha shape vertices
    from scipy.spatial.distance import cdist
    distances = cdist(poisson_vertices, alpha_vertices)
    min_distances = np.min(distances, axis=1)
    
    # Additional mask: keep vertices close to alpha shape
    distance_threshold = np.percentile(min_distances, 80)  # Keep 80% closest
    distance_mask = min_distances <= distance_threshold
    
    # Step 5: Combine filters and remove vertices
    vertices_to_remove = ~(density_mask & distance_mask)
    print(f"Removing {np.sum(vertices_to_remove)}/{len(vertices_to_remove)} vertices")
    
    # Step 6: Apply filtering
    poisson_mesh.remove_vertices_by_mask(vertices_to_remove)
    poisson_mesh.compute_vertex_normals()
    poisson_mesh.vertex_colors = o3d.utility.Vector3dVector(np.empty((0, 3)))

    return poisson_mesh

In [ ]:
# Check out alpha shape
trimmed_mesh = combine_poisson_with_alpha_boundary(leaf_pcd, alpha=0.6)

print("Original points")
o3d.visualization.draw_geometries([leaf_pcd])

print("Trimmed mesh (density + distance filtering)")
o3d.visualization.draw_geometries([trimmed_mesh])

# Optional: visualize with point cloud overlay
print("Improved mesh with original points")
o3d.visualization.draw_geometries([leaf_pcd, trimmed_mesh])

In [ ]:
trimmed_mesh.get_surface_area()

Option 2: based on just distance from the point cloud

In [ ]:
def improve_mesh_boundary_simple(pcd, poisson_depth=9, density_percentile=15, distance_multiplier=2.0):
    """
    Improve Poisson mesh boundary by filtering vertices that are too far from original points
    """
    
    # Step 1: Poisson reconstruction
    print("Creating Poisson mesh...")
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd, depth=poisson_depth
    )
    
    # Step 2: Filter by density
    density_threshold = np.percentile(densities, density_percentile)
    density_mask = densities >= density_threshold
    print(f"Density filtering keeps: {np.sum(density_mask)}/{len(density_mask)} vertices")
    
    # Step 3: Calculate distance threshold based on point cloud spacing
    pcd_points = np.asarray(pcd.points)
    from sklearn.neighbors import NearestNeighbors
    
    # Find average spacing in point cloud
    nbrs = NearestNeighbors(n_neighbors=5, algorithm='kd_tree').fit(pcd_points)
    distances_pcd, _ = nbrs.kneighbors(pcd_points)
    avg_spacing = np.mean(distances_pcd[:, 1])
    distance_threshold = distance_multiplier * avg_spacing
    
    print(f"Average point cloud spacing: {avg_spacing:.6f}")
    print(f"Using distance threshold: {distance_threshold:.6f}")
    
    # Step 4: Filter mesh vertices by distance to original points
    mesh_vertices = np.asarray(mesh.vertices)
    nbrs_mesh = NearestNeighbors(n_neighbors=1, algorithm='kd_tree').fit(pcd_points)
    distances_to_pcd, _ = nbrs_mesh.kneighbors(mesh_vertices)
    distances_to_pcd = distances_to_pcd.flatten()
    
    distance_mask = distances_to_pcd <= distance_threshold
    print(f"Distance filtering keeps: {np.sum(distance_mask)}/{len(distance_mask)} vertices")
    
    # Step 5: Combine filters and remove vertices
    vertices_to_remove = ~(density_mask & distance_mask)
    print(f"Removing {np.sum(vertices_to_remove)}/{len(vertices_to_remove)} vertices")
    
    # Step 6: Apply filtering
    mesh.remove_vertices_by_mask(vertices_to_remove)
    mesh.compute_vertex_normals()
    mesh.vertex_colors = o3d.utility.Vector3dVector(np.empty((0, 3)))

    return mesh, distances_to_pcd

In [ ]:
trimmed_mesh, distances = improve_mesh_boundary_simple(
    leaf_pcd, 
    poisson_depth=7,
    density_percentile=15,
    distance_multiplier=10  # Adjust this to make boundary tighter (smaller) or looser (larger)
)

print("Original points")
o3d.visualization.draw_geometries([leaf_pcd])

print("Trimmed mesh (density + distance filtering)")
o3d.visualization.draw_geometries([trimmed_mesh])

# Optional: visualize with point cloud overlay
print("Trimmed mesh with original points")
o3d.visualization.draw_geometries([leaf_pcd, trimmed_mesh])

In [ ]:
trimmed_mesh.get_surface_area() # This seems to give the most reasonable visual results and plausible area

In [ ]:
success = o3d.io.write_triangle_mesh("trimmed_mesh.ply", trimmed_mesh)

### Trimming based on density + distance + boundary (WIP)

In [ ]:
# idx = 17
# leaf_paths = natsorted(Path("../data/TrackPlant3D/dense_leaves/maize").iterdir())
# leaf_paths[idx]

In [ ]:
#pcd = o3d.t.io.read_point_cloud(str(leaf_paths[idx]))
pcd = o3d.t.geometry.PointCloud(o3d.core.Tensor(leaf_np))
pcd.estimate_normals()
pcd.orient_normals_to_align_with_direction()
pcd.orient_normals_consistent_tangent_plane(k=30)  # This is very important!!

In [ ]:
# Boundary detection (angle of neighbours within a radius, requires normals, and API only exists in tensor-based pcd)
# https://www.open3d.org/docs/latest/python_api/open3d.t.geometry.PointCloud.html#open3d.t.geometry.PointCloud.compute_boundary_points
edge = pcd.compute_boundary_points(radius=10, max_nn=30, angle_threshold=180)
o3d.visualization.draw_geometries([edge[0].to_legacy()])